In [5]:
import sys
!{sys.executable} -m pip install transformers
!{sys.executable} -m pip install torch torchvision torchaudio
!{sys.executable} -m pip install datasets==2.16.1

  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 135.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 119.9 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.6/793.6 kB 25.0 MB/s  0:00:00
Using cached s

**Importing Data**

In [3]:
import datasets

In [4]:
dataset = datasets.load_dataset("m-aliabbas/idrak_timit_subsample1", split="train")

In [5]:
dataset

Dataset({
    features: ['audio', 'transcription'],
    num_rows: 1296
})

In [9]:
!{sys.executable} -m pip install librosa soundfile

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached soundfile-0.13.1-py2.py3-none-manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached numba-0.65.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.9 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached soxr-1.0.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached llvmlite-0.47.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.0 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
Using cached soundfile-0.13.1-py2.py3-none-manylinux_2_28_x86_64.whl (1.3 MB)
Using cached audioread-3.1.0-py3-none-any.whl (23 kB)
Using cached numba-0.65.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.7 MB)
Using cached llvmlite-0.47.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (56.3 MB)
Using cached pooch-1.9.0-

**Looking into 1st data point**

In [10]:
from IPython.display import Audio
data = next(iter(dataset))
audio = data['audio']["array"]
sr = data['audio']["sampling_rate"]
print(data['transcription'])
Audio(audio, rate=sr)

don t ask me to carry an oily rag like that


**Tokenize using Character level encoding**

In [12]:
from tokenizers import Tokenizer, pre_tokenizers, decoders, models
def get_tokenizer():
  tokenizer = Tokenizer(models.BPE())
  tokenizer.add_special_tokens(["!"])
  tokenizer.add_tokens(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ '"))
  tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
  tokenizer.decoder = decoders.ByteLevel()
  tokenizer.blank_token = tokenizer.token_to_id("!")
  return tokenizer

In [13]:
tokenizer = get_tokenizer()
tokens = tokenizer.encode("HELLO WORLD")
tokens.ids

[8, 5, 12, 12, 15, 27, 23, 15, 18, 12, 4]

In [14]:
from torch.utils.data import DataLoader, Dataset
class CommonVoiceDataset(Dataset):
  def __init__(self, voiceset, tokenizer):
    self.dataset = voiceset
    self.num_examples = len(voiceset)
    self.tokenizer = tokenizer

  def __len__(self):
    return self.num_examples

  def __getitem__(self, index):
    item = self.dataset[index]
    waveform = torch.from_numpy(item['audio']["array"]).float()
    text = item['transcription'].upper()
    encoded = self.tokenizer.encode(text)
    return {"audio": waveform, "text": text, "input_ids": torch.tensor(encoded.ids)}


In [15]:
import torch
import torch.nn.functional as F;
def collate_fn(batch):
  max_audio_len = max([len(item["audio"]) for item in batch])
  max_input_ids = max([len(item["input_ids"]) for item in batch])

  audio_tensor = torch.stack(
      [
          F.pad(item["audio"], (0, max_audio_len - len(item["audio"])))
          for item in batch
      ])

  input_ids = torch.stack(
      [
          F.pad(item["input_ids"], (0, max_input_ids - len(item["input_ids"])))
          for item in batch
      ])

  output_dict = {
      "audio": audio_tensor,
      "text": [item["text"] for item in batch],
      "input_ids": input_ids
  }

  return output_dict


In [16]:
def getDataSet():
  dataset = datasets.load_dataset("m-aliabbas/idrak_timit_subsample1", split="train")
  tokenizer = get_tokenizer()
  dataset = CommonVoiceDataset(dataset, tokenizer)
  loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
  return loader

In [17]:
loader = getDataSet()
for batch in loader:
  audio = batch["audio"]
  input_ids = batch["input_ids"]
  print(audio.shape)
  print(input_ids.shape)
  break

torch.Size([32, 82637])
torch.Size([32, 78])


***Downsampling waveform audio***

In [18]:
import torch
import torch.nn as nn
class ResidualDownampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, kernels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernels, padding="same")
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernels, stride=stride)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x) + x
        x = self.conv2(x)
        return x

In [19]:
class DownSamplingNetwork(nn.Module):

    def __init__(self,in_channels=1,hidden_dim=64, embedding_dim=128,kernel_size=2,strides=[3,4,5]):
        super().__init__()

        self.layers = nn.ModuleList()

        self.min_pooling_layer = nn.AvgPool1d(kernel_size=kernel_size)

        for i in range(len(strides)):
            inc = in_channels if i == 0 else hidden_dim
            # downsample input audio one stride by another
            layer = ResidualDownampleBlock(inc,hidden_dim, strides[i], kernels=8)
            self.layers.append(layer)
        self.final = nn.Conv1d(hidden_dim, embedding_dim, kernel_size=4, padding="same")

    def forward(self, x):
        x =  self.min_pooling_layer(x)
        for layer in self.layers:
            x = layer(x)
        x = self.final(x)
        x = x.transpose(1,2)
        return x

***Tranformer Self Attention***

In [23]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F

In [20]:
def calculateAttention(query, key, value):
    weights = torch.matmul(query, key.transpose(-1,-2))
    weights = weights / math.sqrt(key.shape[-1])
    attention_weights = F.softmax(weights, dim=-1)
    score = torch.matmul(attention_weights, value)
    return score

In [21]:
class FeedForward(nn.Module):

    def __init__(self, embeddings):
        super().__init__()
        self.layer1 = nn.Linear(embeddings,embeddings)
        self.layer2 = nn.Linear(embeddings, embeddings)
    
    def forward(self, x):
        x = self.layer1(x)
        x = F.gelu(x)
        x = self.layer2(x)
        return x

In [22]:
class SelfAttention(nn.Module):

    def __init__(self, embeddings):
        super().__init__()
        self.query = nn.Linear(embeddings,embeddings)
        self.key = nn.Linear(embeddings,embeddings)
        self.value = nn.Linear(embeddings,embeddings)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        return calculateAttention(q,k,v)

In [23]:
class TransformerBlock(nn.Module):
    def __init__(self, embeddings):
        super().__init__()
        self.attention = SelfAttention(embeddings)
        self.mlp = FeedForward(embeddings)
        self.batchNorm = nn.LayerNorm(embeddings)
    
    def forward(self, y):
        x = self.attention(y)
        x = self.batchNorm(x)
        x = self.mlp(x)
        x = F.gelu(x)
        return y+x

In [24]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_seq_length):
        super().__init__()
        position = torch.arange(max_seq_length).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, embed_size, 2) * (-math.log(10000.0) / embed_size))

        pe = torch.zeros(max_seq_length, embed_size)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('positional_embedding', pe)

    def forward(self, x):
        return x + self.positional_embedding[:, :x.size(1), :]

In [25]:
class Transformer(nn.Module):
    def __init__(self, embeddings, num_layers, max_seq_length):
        super().__init__()
        self.transformer_blocks = nn.ModuleList([TransformerBlock(embeddings) for _ in range(num_layers)])
        self.pos_encoding = SinusoidalPositionalEncoding(embeddings, max_seq_length)
    
    def forward(self, x):
        x = self.pos_encoding(x)
        for layer in self.transformer_blocks:
            x = layer(x)
        
        return x

**Residual Vector Quantization**

In [27]:
class VectorQuantizater(nn.Module):

    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        # code book vectors
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        nn.init.uniform(self.embedding.weight, -0.1, 0.1)

    def forward(self, x):
        batch, seqLength,embeddingDim = x.shape
        flat_x = x.reshape(batch*seqLength, embeddingDim)

        distances = torch.cdist(flat_x, self.embedding.weight, p=2)
        encoding_index = torch.argmin(distances, dim=1)

        quantized_vector = self.embedding(encoding_index).view(batch, seqLength, embeddingDim)

        # update gradients of previous conv layers and transfomer such that x is closer to code book entry
        e_latent_loss = torch.mean((quantized_vector.detach() - x) ** 2)

        # update code book vector such that its closer to input x
        q_latent_loss = torch.mean((quantized_vector - x.detach()) ** 2)

        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        # straight through estimator
        # quantized is obatined from torch.argmin and derivate of it is 0
        # for gradients to flow backward, we do hack as if nothing got quantized
        # only x will be quantized, i.e. identity, input is quantized as if nothing happened to it
        # codebook vector is learned using above losses 
        quantized_vector = x + (quantized_vector-x).detach()

        return quantized_vector, loss

In [28]:
class ResidualVectorQuanitzer(nn.Module):
    def __init__(self, num_codebooks, codebook_size, embedding_dim):
        super().__init__()

        self.codebooks = nn.ModuleList(
            [
                VectorQuantizater(codebook_size, embedding_dim, commitment_cost=0.3)  for _ in range(num_codebooks)
            ])
    
    def forward(self, x):
        out = 0
        total_loss = 0

        # run through rach of the codebooks and collect nearest codebook vector sum them
        for codebook in self.codebooks:
            output,loss = codebook(x)
            x = x-output
            out = out+output
            total_loss += loss
        
        return out,total_loss

In [29]:
class TranscribeModel(nn.Module):

    def __init__(self, num_codebooks, codebook_size, embedding_dim, vocab_size, strides, kernel_size, num_transformer_layers, max_seq_len):
        super().__init__()
        
        self.num_codebooks = num_codebooks
        self.codebook_size = codebook_size
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.strides = strides
        self.kernel_size = kernel_size
        self.num_trasnformer_layers = num_transformer_layers
        self.max_seq_len = max_seq_len

        hidden = embedding_dim // 2
        self.downSamplingLayer = DownSamplingNetwork(in_channels=1, hidden_dim = hidden, embedding_dim=embedding_dim, kernel_size=kernel_size, strides=strides)

        self.transformer = Transformer(embedding_dim, num_transformer_layers, max_seq_len)

        self.rvq = ResidualVectorQuanitzer(num_codebooks, codebook_size, embedding_dim)

        self.outputlayer = nn.Linear(embedding_dim, vocab_size)

    
    def forward(self, x):
        loss = torch.tensor(0.0)
        x = x.unsqueeze(1)
        # downsample audio vector
        x = self.downSamplingLayer(x)
        # contextualize audio frames
        x = self.transformer(x)
        # feed it to rvq
        x,loss = self.rvq(x)
        # map output to mlp
        x = self.outputlayer(x)
        # take softmax to get logits
        # torch.softmax from 0 to 1
        # torch.log_softmax from -inf to 0
        # CTCLoss works better with log since values will be more stable instead of small decimals
        x = torch.log_softmax(x, dim=-1)
        return x,loss

In [31]:
import math
model = TranscribeModel(num_codebooks=3, codebook_size=64, embedding_dim=64, vocab_size=len(tokenizer.get_vocab()), strides=[6,8,4,2], kernel_size=4, max_seq_len=2000, num_transformer_layers=2)

/tmp/ipykernel_3120/601073018.py:11: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  nn.init.uniform(self.embedding.weight, -0.1, 0.1)


In [32]:
loader = getDataSet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [33]:
vq_initial_weight = 10
vq_final_loss_weight = 0.5
vq_warmup_steps = 1000
steps = 0

In [38]:
def loss_fn(log_probs, target, blank_token):
    loss = nn.CTCLoss(blank=blank_token, zero_infinity=True)
    input_lengths = tuple(log_probs.shape[1] for _ in range(log_probs.shape[0]))
    target_lengths = (target != blank_token).sum(dim=1)
    target_lengths = tuple(t.item() for t in target_lengths)

    input_seq_first = log_probs.permute(1,0,2)
    
    # CTC loss expects (time, batch, vocab) instead of (batch, time, vocab)
    # target_lengths = till where real character exist with out padding 0
    # ideally input_lengths should include length with out 0 padding  but
    # Ctc is intelligent to map o to blank tokens
    loss_val = loss(input_seq_first, target, input_lengths, target_lengths)
    return loss_val

In [ ]:
for i in range(500):
    for idx, batch in enumerate(loader):
        audio = batch["audio"]
        target = batch["input_ids"]
        text = batch["text"]

        # if audio is longer than text, add 0's
        if target.shape[1] > audio.shape[1]:
            audio = nn.functional.pad(audio, (0,0,0,target.shape[1]-audio.shape[1]))
        
        optimizer.zero_grad()
        output, vq_loss = model(audio)
        blank_token = tokenizer.token_to_id("!")

        # output  torch.Size([32, 50, 29])
        # target  torch.Size([32, 78])
        ctc_loss = loss_fn(output, target, blank_token) 

        # vq_initial_weight - intially vq loss has precedence over ctc so codebooks stabilize
        # vq_final_loss_weight - final weightage, ctc will have high precedence
        # steps / vq_warmup_steps - will go from 0 to 1
        # this technique helps reduce importance on vq loss compared to ctc
        vq_loss_wt = max(vq_final_loss_weight, vq_initial_weight - (vq_initial_weight - vq_final_loss_weight) * (steps/vq_warmup_steps))

        if vq_loss is None:
            lossval = ctc_loss
        else:
            lossval = ctc_loss + vq_loss_wt * vq_loss

        # weights will be nan if loss becomes infinity, skip this batch
        if torch.isinf(lossval):
            continue
        
        lossval.backward()

        # solve exploding gradients
        # if combined magnitude of all gradients is greater than 10, it scales down
        # so that sum of them will equal to 10
        nn.utils.clip_grad_norm(model.parameters(), max_norm=10.0)

        optimizer.step()

        steps += 1

In [ ]:
save_path = "speech_model.pth"
torch.save(model.state_dict(), save_path)

In [48]:
state_dict = torch.load("speech_model.pth", weights_only=True)
loadedModel = TranscribeModel(num_codebooks=3, codebook_size=64, embedding_dim=64, vocab_size=len(tokenizer.get_vocab()), strides=[6,8,4,2], kernel_size=4, max_seq_len=2000, num_transformer_layers=2) 
loadedModel.load_state_dict(state_dict)

/tmp/ipykernel_13672/601073018.py:11: FutureWarning: `nn.init.uniform` is now deprecated in favor of `nn.init.uniform_`.
  nn.init.uniform(self.embedding.weight, -0.1, 0.1)


<All keys matched successfully>

In [49]:
loader = getDataSet()
for batch in loader:
  audio = batch["audio"]
  input_ids = batch["input_ids"]
  print(audio.shape)
  print(input_ids.shape)

  out = loadedModel(audio)
  print(out[0].shape)
  break

torch.Size([32, 82637])
torch.Size([32, 78])
torch.Size([32, 50, 29])


In [50]:
# remove blanks and repetetions, recontruct predicted sentence
def ctc_decode(sequence, blank):
    res = []
    prev_item = None
    for item in sequence:
        if item != prev_item:  # Remove consecutive duplicates
            if item != blank:  # Remove blank tokens
                res.append(item)
        prev_item = item
    return res

In [54]:
for i in range(32):
    textTokens = input_ids[i]
    predictedTokens = out[0][i]
    ret = torch.argmax(predictedTokens, dim=-1)
    ret = ctc_decode(ret, blank=tokenizer.token_to_id("!"))
    print(f" Actual: {tokenizer.decode(textTokens.tolist())} \n Predicted:{tokenizer.decode(ret)} \n")


 Actual: CLIFF S DISPLAY WAS MISPLACED ON THE SCREEN 
 Predicted:CLIFF S DISPLAY WNS MISPLACED ECES
 Actual: HE CHUCKLED  THE MEMORY VIVID 
 Predicted:HE CHUCKLED  THE TO TAEETET
 Actual: SHE HAD YOUR DARK SUIT IN GREASY WASH WATER ALL YEAR 
 Predicted:AYBNBEBIBNIBY IO D E T NBOBDBYVE IKEGIHEY
 Actual: CLIFF S DISPLAY WAS MISPLACED ON THE SCREEN 
 Predicted:CLIFF S DISPLAY WNS MISPLACED ECES
 Actual: THE COYOTE  BOBCAT  AND HYENA ARE WILD ANIMALS 
 Predicted:THE COYOTE  BOBCAT  AND HYENA ARE WILD ANIMALS
 Actual: SHE HAD YOUR DARK SUIT IN GREASY WASH WATER ALL YEAR 
 Predicted:ALENE HYNO WOYO EO ETYO EYGN VRDEOE PY ED
 Actual: AMOEBAS CHANGE SHAPE CONSTANTLY 
 Predicted:AMOEBAS CHANGE SHAPE CONSANLS
 Actual: SHE HAD YOUR DARK SUIT IN GREASY WASH WATER ALL YEAR 
 Predicted:AYBNBEBIBNIBY IO D E T NBOBDBYVE IKEGIHEY
 Actual: IN MOST DISCUSSIONS OF THIS PHENOMENON  THE FIGURES ARE SUBSTANTIALLY INFLATED 
 Predicted:LIEEO OY I T BOE ONO VBRVEOE HIOH LY
 Actual: ANY RETALIATORY GAS ATTACK WO